In the quick start we'll show you know to build a simple LLM application with LangChain.This application will translate text from English into another- it's just a single LLM call plus some prompting. Still this is a great way to get started with LangChain - a lot feature can be built with just some prompting and LLM call!

after seeing this video , you'll have a high level overview of:
<li>Using Language models</li>
<li>Using Prompt Templates and OutputParsers</li>
<li>Using Langchain Expression Language(LCEL) to chain components  together</li>
<li>Debugging and tracing your application using LangSmith </li>
<li>Deploying your application with LangServe</li>

In this section we'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.
Note that this chat that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:
<li>Conversational RAG: Enable a chatbot experience over an external source of data</li>
<li>Agents: Build a chatbot that can take actions</li>


In [19]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [20]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "Qwen/Qwen3.6-27B" , groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '0.3.27'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001AC0F0B3400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AC0F0B39D0>, model_name='Qwen/Qwen3.6-27B', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [21]:
from langchain_core.messages import HumanMessage
result = model.invoke([HumanMessage(content="Hi , my name is Muskan Chauhan and AI Forward Deployed Engineer")])
result

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Name: Muskan Chauhan\n   - Role: AI Forward Deployed Engineer\n   - Greeting: "Hi"\n\n2.  **Identify Key Elements:**\n   - Personal introduction (name + role)\n   - Role implies expertise in AI, deployment, likely working closely with customers/clients to integrate AI solutions\n   - Tone: Professional, friendly\n\n3.  **Determine Response Goals:**\n   - Acknowledge and greet by name\n   - Acknowledge the role positively\n   - Offer assistance relevant to their role\n   - Keep it professional, concise, and open-ended\n   - Maintain a helpful, collaborative tone\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Muskan! Nice to meet you. AI Forward Deployed Engineer sounds like an exciting and impactful role—bridging cutting-edge AI with real-world customer solutions. How can I assist you today? Whether it\'s architecture advice, deployment strategies, model optimization, debugging, or just

In [22]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Name: Muskan Chauhan\n   - Role: AI Forward Deployed Engineer\n   - Greeting: "Hi"\n\n2.  **Identify Key Elements:**\n   - Personal introduction (name + role)\n   - Role implies expertise in AI, deployment, likely working closely with customers/clients to integrate AI solutions\n   - Tone: Professional, friendly\n\n3.  **Determine Response Goals:**\n   - Acknowledge and greet by name\n   - Acknowledge the role positively\n   - Offer assistance relevant to their role\n   - Keep it professional, concise, and open-ended\n   - Maintain a helpful, collaborative tone\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Muskan! Nice to meet you. AI Forward Deployed Engineer sounds like an exciting and impactful role—bridging cutting-edge AI with real-world customer solutions. How can I assist you today? Whether it\'s architecture advice, deployment strategies, model optimization, debugging, or just brainstorming AI 

In [23]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi , My name is Muskan Chauhan "),
    AIMessage(content="Hi Muskan! Nice to meet you. As an AI Forward Deployed Engineer, you\'re clearly working at the exciting intersection of AI development and real-world customer impact. How can I assist you today? Whether it\'s debugging ML pipelines, optimizing model deployments, exploring new AI frameworks, or brainstorming solutions for enterprise AI use cases, I\'m here to help."),
    HumanMessage(content="What kind of salary i am expected in FDE role")
])

 


AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User: "What kind of salary i am expected in FDE role"\n   - Context: User previously introduced herself as "Muskan Chauhan" and I assumed she\'s an "AI Forward Deployed Engineer" (FDE) based on my previous response. Wait, I actually *assumed* that role in my first response without her explicitly stating it. I should acknowledge that and clarify.\n   - Key question: Expected salary for a Forward Deployed Engineer (FDE) role.\n   - Important factors affecting salary: Location, company, experience level, specific domain (AI/ML vs general), education, negotiation, stock/RSUs, bonuses.\n\n2.  **Identify Key Variables for Salary Estimation:**\n   - **Role:** Forward Deployed Engineer (FDE) - typically works at the intersection of engineering, customer success, and AI/ML deployment. Often found at companies like Google, Microsoft, Meta, AWS, etc.\n   - **Location:** Major tech hubs (US, especially B

### Message History
We can use a Message Hsitory class to wrap our model and male it stateful.This will Keep track of inputs and outputs of the model and store them in some datastore. Future Interaction will then load those message and pass them into chain as pat of the input , Let's see how to use this !

In [24]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}


def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model , get_session_history)


e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [25]:
config = {"configurable":{"session_id":"chat1"}}

In [26]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer")
    ],
    config=config
)

In [ ]:
## Chnage the config  --> session id
config1 = {"configurable":{"session_id":"chat1"}}
response = with_message_history.invoke(
    [
        HumanMessage(content="Whats my name")
    ],
    config=config1
)
response.content


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "Whats my name"\n2.  **Identify Key Constraint:** As an AI, I don\'t have memory of past conversations unless explicitly provided in the current session/context. I also don\'t have access to personal information about users unless they\'ve shared it in this conversation.\n3.  **Determine Response Strategy:** \n   - Acknowledge the question directly\n   - Explain that I don\'t know their name\n   - Clarify why (no access to personal data/memory across sessions)\n   - Offer to remember it for this conversation if they share it\n   - Keep it friendly and concise\n4.  **Draft Response (mental):** I don\'t actually know your name! I don\'t have access to personal information or memory from past conversations. If you\'d like to tell me your name, I\'ll be happy to use it during our chat. What should I call you?\n5.  **Refine Response:** Check against guidelines: accurate, clear, polite, acknowledges AI limit